# LangChain Fundamentals: Part 3
## Retrieval Augmented Generation (RAG)

### Learning Objectives
By the end of this notebook, you'll understand:
1. **What is RAG?** - Retrieving relevant data and using it as context
2. **Vector Stores** - Converting text to numbers (embeddings) for similarity search
3. **Similarity Search** - Finding relevant documents for a query
4. **RAG Pattern** - Retrieve → Augment → Generate workflow
5. **When to Use RAG** - Real-world applications and benefits

## The Problem: Limited Knowledge

### Without RAG:
```python
Q: "Tell me about company ABC"
A: "I don't know about company ABC. My training data ends in April 2024."
```

### With RAG:
```python
Q: "Tell me about company ABC"
A: "Based on the documents provided, company ABC was founded in 2020..."
```

## What is RAG?

**RAG = Retrieval Augmented Generation**

Instead of asking the LLM from scratch, we:
1. **Retrieve** relevant documents from a database
2. **Augment** the prompt with those documents as context
3. **Generate** an answer using that context

### Why RAG?
✅ **More accurate** - Uses real data, not just training data  
✅ **Up-to-date** - Can include recent information  
✅ **Verifiable** - You can check the sources  
✅ **Cost-effective** - Cheaper than fine-tuning  
✅ **Controllable** - You decide what data the AI sees

## Setup: Environment and Models

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Initialize the LLM and embeddings
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

# Embeddings convert text to numerical vectors
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("✓ LLM and embeddings initialized!")

## Step 1: Loading Your Data

First, we need some data to work with. This could be:
- Excel files
- PDFs
- Database records
- Web pages
- Any text data

In this example, we'll load data from an Excel file.

In [ ]:
import pandas as pd

# Load data from Excel file
# Make sure you have 'pitti.xlsx' in your directory
data = pd.read_excel('pitti.xlsx')

print(f"Loaded {len(data)} records")
print(f"\nColumns: {list(data.columns)}")
print(f"\nFirst row:")
print(data.iloc[0])

## Step 2: Understanding Embeddings

### What are Embeddings?

Embeddings convert text into **vectors** (lists of numbers) that capture the meaning of the text.

```python
Text: "red car"
Embedding: [0.234, -0.123, 0.987, ..., 0.456]  # 1536 numbers

Text: "blue car"
Embedding: [0.245, -0.110, 0.992, ..., 0.461]  # Similar numbers

Text: "pizza"
Embedding: [-0.456, 0.789, 0.123, ..., 0.234]  # Very different numbers
```

### Key Idea:
- **Similar text** → Similar embeddings
- **Different text** → Different embeddings

This allows us to find similar documents using **distance calculations**!

## Step 3: Converting Data to Documents

LangChain uses a `Document` object to represent pieces of text. Let's convert our data:

In [ ]:
from langchain_core.documents import Document

# Create a list to hold our documents
documents = []

# Convert each row's description to a Document
for idx, desc in enumerate(data['description']):
    doc = Document(
        page_content=str(desc),
        metadata={'index': idx}  # Store the row index for reference
    )
    documents.append(doc)

# For this example, let's use just the first 50 documents
documents = documents[:50]

print(f"Created {len(documents)} documents")
print(f"\nFirst document:")
print(f"Content: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

## Step 4: Creating a Vector Store

### What is a Vector Store?

A vector store is a database that:
1. **Stores embeddings** - Converts documents to vectors
2. **Indexes them** - Organizes for fast searching
3. **Calculates similarity** - Finds the most similar documents to a query

### Popular Vector Stores:
- **FAISS** (Free, local, fast) - What we'll use
- **Pinecone** (Cloud-based, scalable)
- **Weaviate** (Open-source, powerful)
- **Chroma** (Simple, embedding-native)

In [ ]:
import warnings
warnings.filterwarnings('ignore', message=".*langchain-community.*")

from langchain_community.vectorstores.faiss import FAISS

# Create a FAISS vector store from our documents
# This step:
# 1. Converts each document to an embedding using OpenAI's embeddings model
# 2. Stores them in a searchable index
vector_store = FAISS.from_documents(documents, embeddings)

print("✓ Vector store created!")
print(f"  Stored {len(documents)} documents")
print(f"  Each embedding has {len(embeddings.embed_query('test'))} dimensions")

## Step 5: Similarity Search

Now we can search for documents similar to a query!

In [ ]:
# Define a query
query = "Italian leather craftsmanship"

print(f"Query: {query}\n")
print("="*80)

# Search for the 5 most similar documents
retrieved_docs = vector_store.similarity_search(query, k=5)

print(f"\nFound {len(retrieved_docs)} similar documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Result {i}:")
    print(f"  {doc.page_content[:150]}...")
    print(f"  Metadata: {doc.metadata}")
    print()

## Step 6: Understanding Similarity Scores

We can also get similarity scores to see how relevant each result is:

In [ ]:
# Search with similarity scores
# Lower distance = more similar
docs_with_scores = vector_store.similarity_search_with_score(query, k=5)

print(f"Query: {query}\n")
print("Document Relevance Scores:")
print("(Lower score = more similar)\n")

for i, (doc, score) in enumerate(docs_with_scores, 1):
    print(f"Result {i}: Score = {score:.4f}")
    print(f"  {doc.page_content[:100]}...\n")

## Step 7: The RAG Pattern - Retrieve

Now let's implement the full RAG pattern:

### Step 1: Retrieve
Get relevant documents from the vector store

In [ ]:
query = "Which brand specializes in leather jackets?"

# STEP 1: Retrieve
# Get the k most similar documents
k = 5
retrieved_docs = vector_store.similarity_search(query, k=k)

# Combine all the retrieved content into one big context
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

print("Retrieve Step:")
print(f"  Query: {query}")
print(f"  Retrieved {len(retrieved_docs)} documents")
print(f"  Context length: {len(context)} characters")
print(f"\nContext preview:\n{context[:300]}...")

## Step 8: The RAG Pattern - Augment & Generate

### Step 2: Augment
Add the context to the prompt

### Step 3: Generate
Let the LLM answer using the context

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# STEP 2 & 3: Augment + Generate
# Create a prompt that includes context
prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about fashion brands.

Answer the user's question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {query}

Answer:
""")

# Create the chain
parser = StrOutputParser()
chain = prompt_template | llm | parser

# Generate the answer
answer = chain.invoke({
    'context': context,
    'query': query
})

print("Answer:")
print(answer)

## Step 9: Building a Reusable RAG Function

Let's create a function that packages the entire RAG pipeline:

In [ ]:
def rag_query(query, k=5, vector_store=vector_store, llm=llm):
    """
    Perform RAG (Retrieval Augmented Generation) on a query.
    
    Args:
        query: The user's question
        k: Number of documents to retrieve (default 5)
        vector_store: The FAISS vector store to search
        llm: The language model to use for generation
    
    Returns:
        The generated answer based on retrieved context
    """
    # STEP 1: Retrieve
    retrieved_docs = vector_store.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # STEP 2 & 3: Augment + Generate
    prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about fashion brands and companies.
Answer the user's question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {query}
Answer:
""")
    
    parser = StrOutputParser()
    chain = prompt_template | llm | parser
    
    answer = chain.invoke({
        'context': context,
        'query': query
    })
    
    return answer

print("✓ RAG function created!")

## Step 10: Testing the RAG Function

Let's test our RAG pipeline with different queries:

In [ ]:
# Test different queries
test_queries = [
    "Tell me about Made in Italy brands",
    "Which brands use innovation?",
    "Are there any family businesses?",
    "Which brands are specialized in jackets?",
    "Are there any US-based brands?"
]

print("RAG Query Results:\n")
print("="*80)

for query in test_queries:
    print(f"\nQ: {query}")
    answer = rag_query(query, k=5)
    print(f"A: {answer}")
    print("-"*80)

## How RAG Compares to Direct LLM Calls

### Without RAG:
```python
response = llm.invoke("Tell me about company ABC from the pitti dataset")
# ❌ Result: Generic answer, may contain hallucinations
```

### With RAG:
```python
retrieved_docs = vector_store.similarity_search("company ABC")
response = llm.invoke(f"Based on: {context}, answer: {query}")
# ✅ Result: Accurate, grounded answer with real data
```

## Real-World RAG Applications

### 1. **Customer Support Chatbot**
- Retrieve FAQ documents
- Generate helpful responses based on company knowledge

### 2. **Document Search Engine**
- Index all company documents
- Answer questions about them

### 3. **Research Assistant**
- Index research papers
- Synthesize answers from multiple sources

### 4. **Legal Document Analysis**
- Index contracts and regulations
- Answer compliance questions

### 5. **Product Recommendation**
- Index product catalog
- Recommend based on customer queries

## Advanced: Saving and Loading Vector Stores

For production, you'll want to save your vector store so you don't have to recreate it every time:

In [ ]:
# Save the vector store
vector_store.save_local("faiss_index")
print("✓ Vector store saved to 'faiss_index' directory")

# Load it back
from langchain_community.vectorstores import FAISS
loaded_vs = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
print("✓ Vector store loaded!")

# Test with loaded vector store
test_answer = rag_query("What brands create tailored clothing?", vector_store=loaded_vs)
print(f"\nTest query result: {test_answer[:100]}...")

## Summary: The RAG Pattern

### Three Simple Steps:

```python
# 1. RETRIEVE - Find relevant documents
docs = vector_store.similarity_search(query, k=5)

# 2. AUGMENT - Add documents to prompt
context = "\n\n".join([doc.page_content for doc in docs])
prompt = f"Given: {context}, answer: {query}"

# 3. GENERATE - Get answer from LLM
answer = llm.invoke(prompt)
```

### Key Takeaways:
- ✅ **More accurate** than LLM alone
- ✅ **Verifiable** - you know the sources
- ✅ **Scalable** - works with large datasets
- ✅ **Cost-effective** - no need for fine-tuning
- ⚠️ **Retrieval quality matters** - bad retrieval = bad answers

## Practice Exercises

### Exercise 1: Query Variations
Test the RAG system with 5 different queries about the brands in the dataset.

### Exercise 2: Adjust k Parameter
Try changing `k` (number of documents retrieved) from 3 to 10. How does the answer quality change?

### Exercise 3: Better Prompts
Modify the prompt template to get more structured answers (e.g., bullet points).

### Exercise 4: Similarity Scores
Print similarity scores for each retrieved document and analyze how they relate to answer quality.

### Exercise 5: Different Data
Try loading a different Excel file and building a RAG system for it.